In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window as W

In [0]:
CATALOG_BRONZE = 'mini_project'
SCHEMA_BRONZE = 'bronze_layer'

CATALOG_SILVER = 'mini_project'
SCHEMA_SILVER = 'silver_layer'

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_SILVER}.{SCHEMA_SILVER}")

In [0]:
# purchase_order_detail_df = spark.table(f"{CATALOG}.{SCHEMA}.purchasing_purchaseorderdetail")
# purchase_order_header_df = spark.table(f"{CATALOG}.{SCHEMA}.purchasing_purchaseorderheader")
# product_df = spark.table(f"{CATALOG}.{SCHEMA}.production_product")
# product_subcategory_df = spark.table(f"{CATALOG_SILVER}.{SCHEMA_BRONZE}.production_productsubcategory")
# product_category_df = spark.table(f"{CATALOG_SILVER}.{SCHEMA_BRONZE}.production_productcategory")

In [0]:
def dedupe_by_column_as_id(df: DataFrame, column: str) -> DataFrame:
    """
    Deduplicate records by column.
    Keeps the latest record based on modifieddate.
    """

    w = W.partitionBy(column).orderBy(F.col("modifieddate").desc_nulls_last())

    return (
        df
        .withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

In [0]:

def cast_purchaseorderdetail_types(df: DataFrame) -> DataFrame:
    # Handles strings like "2022-04-29T00:00:00.000Z"
    return (
        df
        .withColumn("purchaseorderid", F.col("purchaseorderid").cast(T.LongType()))
        .withColumn("purchaseorderdetailid", F.col("purchaseorderdetailid").cast(T.LongType()))
        .withColumn("productid", F.col("productid").cast(T.LongType()))
        .withColumn("orderqty", F.col("orderqty").cast(T.LongType()))
        .withColumn("unitprice", F.col("unitprice").cast(T.DecimalType(18, 5)))
        .withColumn("receivedqty", F.col("receivedqty").cast(T.LongType()))
        .withColumn("rejectedqty", F.col("rejectedqty").cast(T.LongType()))
        .withColumn("duedate", F.to_timestamp("duedate"))          # parses ISO-like timestamps
        .withColumn("modifieddate", F.to_timestamp("modifieddate"))
    )

def rename_purchaseorderdetail_columns(df: DataFrame) -> DataFrame:
    """
    Static, table-specific column renaming for purchasing_purchaseorderdetail.
    Converts known camelCase columns into snake_case.
    """

    rename_map = {
        "purchaseorderid": "purchase_order_id",
        "purchaseorderdetailid": "purchase_order_detail_id",
        "productid": "product_id",
        "orderqty": "order_qty",
        "receivedqty": "received_qty",
        "rejectedqty": "rejected_qty",
        "unitprice": "unit_price",
        "duedate": "due_date",
        "modifieddate": "modified_date"
    }

    renamed_df = df
    for old_name, new_name in rename_map.items():
        if old_name in renamed_df.columns:
            renamed_df = renamed_df.withColumnRenamed(old_name, new_name)

    return renamed_df

def clean_purchaseorderdetail_values(df: DataFrame) -> DataFrame:
    # Basic sanity cleanup:
    # - trim impossible negatives (keep original in flags)
    # - normalize nulls for qty fields
    return (
        df
        .withColumn("orderqty", F.when(F.col("orderqty") < 0, F.lit(0)).otherwise(F.col("orderqty")))
        .withColumn("unitprice", F.when(F.col("unitprice") < 0, F.lit(0)).otherwise(F.col("unitprice")))
        .withColumn("receivedqty", F.coalesce(F.col("receivedqty"), F.lit(None).cast(T.DecimalType(18, 4))))
        .withColumn("rejectedqty", F.coalesce(F.col("rejectedqty"), F.lit(None).cast(T.DecimalType(18, 4))))
    )

In [0]:
purchase_order_detail_df = (
    spark
    .table(f"{CATALOG_BRONZE}.{SCHEMA_BRONZE}.purchasing_purchaseorderdetail")
    .transform(cast_purchaseorderdetail_types)
    .transform(lambda df: dedupe_by_column_as_id(df, "purchaseorderdetailid"))
    .transform(clean_purchaseorderdetail_values)
    .transform(rename_purchaseorderdetail_columns)
    .write.mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.purchase_order_detail")
) 


In [0]:
def cast_purchaseorderheader_types(df: DataFrame) -> DataFrame:
    """
    Type casting for purchasing_purchaseorderheader.
    """

    return (
        df
        # ---- IDs / ints ----
        .withColumn("purchaseorderid", F.col("purchaseorderid").cast(T.LongType()))
        .withColumn("revisionnumber", F.col("revisionnumber").cast(T.IntegerType()))
        .withColumn("status", F.col("status").cast(T.IntegerType()))
        .withColumn("employeeid", F.col("employeeid").cast(T.LongType()))
        .withColumn("vendorid", F.col("vendorid").cast(T.LongType()))
        .withColumn("shipmethodid", F.col("shipmethodid").cast(T.LongType()))
        # ---- Timestamps ----
        .withColumn("orderdate", F.to_timestamp("orderdate"))
        .withColumn("shipdate", F.to_timestamp("shipdate"))
        .withColumn("modifieddate", F.to_timestamp("modifieddate"))
        # ---- Money ----
        .withColumn("subtotal", F.col("subtotal").cast(T.DecimalType(18, 4)))
        .withColumn("taxamt", F.col("taxamt").cast(T.DecimalType(18, 4)))
        .withColumn("freight", F.col("freight").cast(T.DecimalType(18, 4)))
    )

def rename_purchaseorderheader_columns(df: DataFrame) -> DataFrame:
    """
    Static column renaming for purchasing_purchaseorderheader (bronze -> silver).
    """

    rename_map = {
        "purchaseorderid": "purchase_order_id",
        "revisionnumber": "revision_number",
        "status": "status",
        "employeeid": "employee_id",
        "vendorid": "vendor_id",
        "shipmethodid": "ship_method_id",
        "orderdate": "order_date",
        "shipdate": "ship_date",
        "subtotal": "subtotal",
        "taxamt": "tax_amt",
        "freight": "freight",
        "modifieddate": "modified_date"
    }

    out = df
    for old_name, new_name in rename_map.items():
        if old_name in out.columns:
            out = out.withColumnRenamed(old_name, new_name)

    return out

In [0]:
purchase_order_header_df = (
    spark
    .table(f"{CATALOG_BRONZE}.{SCHEMA_BRONZE}.purchasing_purchaseorderheader")
    .transform(cast_purchaseorderheader_types)
    .transform(lambda df: dedupe_by_column_as_id(df, "purchaseorderid"))
    .transform(rename_purchaseorderheader_columns)
    .write.mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.purchase_order_header")
) 


In [0]:
def cast_product_types(df: DataFrame) -> DataFrame:
    """
    Type casting for product table (bronze -> silver).
    No new columns, no cleaning logic.
    """

    return (
        df
        .withColumn("productid", F.col("productid").cast(T.LongType()))
        .withColumn("name", F.col("name").cast(T.StringType()))
        .withColumn("productnumber", F.col("productnumber").cast(T.StringType()))
        .withColumn("makeflag", F.col("makeflag").cast(T.BooleanType()))
        .withColumn("finishedgoodsflag", F.col("finishedgoodsflag").cast(T.BooleanType()))
        .withColumn("color", F.col("color").cast(T.StringType()))
        .withColumn("safetystocklevel", F.col("safetystocklevel").cast(T.IntegerType()))
        .withColumn("reorderpoint", F.col("reorderpoint").cast(T.IntegerType()))
        .withColumn("standardcost", F.col("standardcost").cast(T.DecimalType(18, 4)))
        .withColumn("listprice", F.col("listprice").cast(T.DecimalType(18, 4)))
        .withColumn("size", F.col("size").cast(T.StringType()))
        .withColumn("sizeunitmeasurecode", F.col("sizeunitmeasurecode").cast(T.StringType()))
        .withColumn("weightunitmeasurecode", F.col("weightunitmeasurecode").cast(T.StringType()))
        .withColumn("weight", F.col("weight").cast(T.DecimalType(18, 4)))
        .withColumn("daystomanufacture", F.col("daystomanufacture").cast(T.IntegerType()))
        .withColumn("productline", F.col("productline").cast(T.StringType()))
        .withColumn("class", F.col("class").cast(T.StringType()))
        .withColumn("style", F.col("style").cast(T.StringType()))
        .withColumn("productsubcategoryid", F.col("productsubcategoryid").cast(T.IntegerType()))
        .withColumn("productmodelid", F.col("productmodelid").cast(T.IntegerType()))
        .withColumn("sellstartdate", F.to_timestamp("sellstartdate"))
        .withColumn("sellenddate", F.to_timestamp("sellenddate"))
        .withColumn("discontinueddate", F.to_timestamp("discontinueddate"))
        .withColumn("rowguid", F.col("rowguid").cast(T.StringType()))
        .withColumn("modifieddate", F.to_timestamp("modifieddate"))
    )

def sanity_clean_product(df: DataFrame) -> DataFrame:
    """
    Light sanity cleaning for product table.
    No new columns. No business logic. Just basic guardrails.
    """

    return (
        df
        # IDs and counts should not be negative
        .withColumn(
            "safetystocklevel",
            F.when(F.col("safetystocklevel") < 0, F.lit(None)).otherwise(F.col("safetystocklevel"))
        )
        .withColumn(
            "reorderpoint",
            F.when(F.col("reorderpoint") < 0, F.lit(None)).otherwise(F.col("reorderpoint"))
        )
        .withColumn(
            "daystomanufacture",
            F.when(F.col("daystomanufacture") < 0, F.lit(None)).otherwise(F.col("daystomanufacture"))
        )
        # Prices and costs should not be negative
        .withColumn(
            "standardcost",
            F.when(F.col("standardcost") < 0, F.lit(None)).otherwise(F.col("standardcost"))
        )
        .withColumn(
            "listprice",
            F.when(F.col("listprice") < 0, F.lit(None)).otherwise(F.col("listprice"))
        )
        # Weight should not be negative
        .withColumn(
            "weight",
            F.when(F.col("weight") < 0, F.lit(None)).otherwise(F.col("weight"))
        )
        # End dates should not be before start dates
        .withColumn(
            "sellenddate",
            F.when(
                F.col("sellenddate").isNotNull() &
                F.col("sellstartdate").isNotNull() &
                (F.col("sellenddate") < F.col("sellstartdate")),
                F.lit(None)
            ).otherwise(F.col("sellenddate"))
        )
        .withColumn(
            "discontinueddate",
            F.when(
                F.col("discontinueddate").isNotNull() &
                F.col("sellstartdate").isNotNull() &
                (F.col("discontinueddate") < F.col("sellstartdate")),
                F.lit(None)
            ).otherwise(F.col("discontinueddate"))
        )
    )

def rename_product_columns(df: DataFrame) -> DataFrame:
    """
    Static column renaming for production_product (bronze -> silver).
    """

    rename_map = {
        "productid": "product_id",
        "name": "name",
        "productnumber": "product_number",
        "makeflag": "make_flag",
        "finishedgoodsflag": "finished_goods_flag",
        "color": "color",
        "safetystocklevel": "safety_stock_level",
        "reorderpoint": "reorder_point",
        "standardcost": "standard_cost",
        "listprice": "list_price",
        "size": "size",
        "sizeunitmeasurecode": "size_unit_measure_code",
        "weightunitmeasurecode": "weight_unit_measure_code",
        "weight": "weight",
        "daystomanufacture": "days_to_manufacture",
        "productline": "product_line",
        "class": "class",
        "style": "style",
        "productsubcategoryid": "product_subcategory_id",
        "productmodelid": "product_model_id",
        "sellstartdate": "sell_start_date",
        "sellenddate": "sell_end_date",
        "discontinueddate": "discontinued_date",
        "rowguid": "row_guid",
        "modifieddate": "modified_date"
    }

    out = df
    for old_name, new_name in rename_map.items():
        if old_name in out.columns:
            out = out.withColumnRenamed(old_name, new_name)

    return out

In [0]:
product_df = (
    spark
    .table(f"{CATALOG_BRONZE}.{SCHEMA_BRONZE}.production_product")
    .transform(cast_product_types)
    .transform(lambda df: dedupe_by_column_as_id(df, "productid"))
    .transform(sanity_clean_product)
    .transform(rename_product_columns)
    .write
    .mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product")
) 


In [0]:
def rename_product_subcategory_cols(df: DataFrame) -> DataFrame:
    """
    Renames columns for the product subcategory table to standardized snake_case names.

    This function is intentionally table-specific and only renames the expected columns
    for the product subcategory dataset.

    Args:
        df (DataFrame): Input Spark DataFrame from the Bronze layer.

    Returns:
        DataFrame: DataFrame with standardized snake_case column names.
    """
    return (
        df
        .withColumnRenamed("productsubcategoryid", "product_subcategory_id")
        .withColumnRenamed("productcategoryid", "product_category_id")
        .withColumnRenamed("name", "name")
        .withColumnRenamed("rowguid", "row_guid")
        .withColumnRenamed("modifieddate", "modified_date")
    )

def cast_product_subcategory_types(df: DataFrame) -> DataFrame:
    """
    Casts columns in the product subcategory DataFrame to their expected data types.

    Expected input columns:
      - productsubcategoryid (int)
      - productcategoryid (int)
      - name (string)
      - rowguid (string / UUID)
      - modifieddate (timestamp)

    This function is intended to be used in the Silver layer to standardize
    raw data ingested from the Bronze layer.

    Args:
        df (DataFrame): Input Spark DataFrame from the Bronze layer.

    Returns:
        DataFrame: DataFrame with columns cast to their correct data types.
    """
    return (
        df
        .withColumn("productsubcategoryid", F.col("productsubcategoryid").cast(T.IntegerType()))
        .withColumn("productcategoryid", F.col("productcategoryid").cast(T.IntegerType()))
        .withColumn("name", F.col("name").cast(T.StringType()))
        .withColumn("rowguid", F.col("rowguid").cast(T.StringType()))
        .withColumn("modifieddate", F.to_timestamp("modifieddate"))
    )

In [0]:
product_subcategory_df = (
    spark
    .table(f"{CATALOG_BRONZE}.{SCHEMA_BRONZE}.production_productsubcategory")
    .transform(cast_product_subcategory_types)
    .transform(lambda df: dedupe_by_column_as_id(df, "productsubcategoryid"))
    .transform(rename_product_subcategory_cols)
    .write.mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product_subcategory")
) 

In [0]:
def cast_product_category_types(df: DataFrame) -> DataFrame:
    """
    Casts columns in the product DataFrame to their expected data types.

    Expected input columns:
      - productcategoryid (int)
      - name (string)
      - rowguid (string / UUID)
      - modifieddate (timestamp)

    This function is intended to be used in the Silver layer to standardize
    raw data ingested from the Bronze layer.

    Args:
        df (DataFrame): Input Spark DataFrame from the Bronze layer.

    Returns:
        DataFrame: DataFrame with columns cast to their correct data types.
    """
    return (
        df
        .withColumn("productcategoryid", F.col("productcategoryid").cast(T.IntegerType()))
        .withColumn("name", F.col("name").cast(T.StringType()))
        .withColumn("rowguid", F.col("rowguid").cast(T.StringType()))
        .withColumn("modifieddate", F.to_timestamp("modifieddate"))
    )

def rename_product_category_cols(df: DataFrame) -> DataFrame:
    """
    Renames columns for the product category table to standardized snake_case names.

    This function is intentionally table-specific and only renames the expected columns
    for the product category dataset.

    Args:
        df (DataFrame): Input Spark DataFrame from the Bronze layer.

    Returns:
        DataFrame: DataFrame with standardized snake_case column names.
    """
    return (
        df
        .withColumnRenamed("productcategoryid", "product_category_id")
        .withColumnRenamed("name", "name")
        .withColumnRenamed("rowguid", "row_guid")
        .withColumnRenamed("modifieddate", "modified_date")
    )

In [0]:
product_category_df = (
    spark
    .table(f"{CATALOG_BRONZE}.{SCHEMA_BRONZE}.production_productcategory")
    .transform(cast_product_category_types)
    .transform(lambda df: dedupe_by_column_as_id(df, "productcategoryid"))
    .transform(rename_product_category_cols)
    .write.mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product_category")
) 

In [0]:
def cast_salesorderdetail_types(df: DataFrame) -> DataFrame:
    """
    Type casting for salesorderdetail.
    No new columns. No cleaning logic.
    """
    return (
        df
        .withColumn("salesorderid", F.col("salesorderid").cast(T.LongType()))
        .withColumn("salesorderdetailid", F.col("salesorderdetailid").cast(T.LongType()))
        .withColumn("carriertrackingnumber", F.col("carriertrackingnumber").cast(T.StringType()))
        .withColumn("orderqty", F.col("orderqty").cast(T.IntegerType()))
        .withColumn("productid", F.col("productid").cast(T.LongType()))
        .withColumn("specialofferid", F.col("specialofferid").cast(T.LongType()))
        .withColumn("unitprice", F.col("unitprice").cast(T.DecimalType(18, 4)))
        .withColumn("unitpricediscount", F.col("unitpricediscount").cast(T.DecimalType(18, 6)))
        .withColumn("rowguid", F.col("rowguid").cast(T.StringType()))
        .withColumn("modifieddate", F.to_timestamp("modifieddate"))
    )


def rename_salesorderdetail_columns(df: DataFrame) -> DataFrame:
    """
    Static column renaming for salesorderdetail (camelCase -> snake_case).
    """
    rename_map = {
        "salesorderid": "sales_order_id",
        "salesorderdetailid": "sales_order_detail_id",
        "carriertrackingnumber": "carrier_tracking_number",
        "orderqty": "order_qty",
        "productid": "product_id",
        "specialofferid": "special_offer_id",
        "unitprice": "unit_price",
        "unitpricediscount": "unit_price_discount",
        "rowguid": "row_guid",
        "modifieddate": "modified_date",
    }

    out = df
    for old_name, new_name in rename_map.items():
        if old_name in out.columns:
            out = out.withColumnRenamed(old_name, new_name)
    return out

def sanity_clean_salesorderdetail(df: DataFrame) -> DataFrame:
    """
    Light sanity cleaning for salesorderdetail.
    """
    return (
        df
        # quantities should not be negative
        .withColumn("orderqty", F.when(F.col("orderqty") < 0, F.lit(0)).otherwise(F.col("orderqty")))
        # prices should not be negative
        .withColumn("unitprice", F.when(F.col("unitprice") < 0, F.lit(0)).otherwise(F.col("unitprice")))
    )

In [0]:
sales_order_detail_df = (
    spark
    .table(f"{CATALOG_BRONZE}.{SCHEMA_BRONZE}.sales_salesorderdetail")
    .transform(cast_salesorderdetail_types)
    .transform(sanity_clean_salesorderdetail)
    .transform(lambda df: dedupe_by_column_as_id(df, "salesorderdetailid"))
    .transform(rename_salesorderdetail_columns)
    .write.mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.sales_order_detail")
) 

In [0]:
def cast_salesorderheader_types(df: DataFrame) -> DataFrame:
    """
    Type casting for salesorderheader.
    """
    return (
        df
        .withColumn("salesorderid", F.col("salesorderid").cast(T.LongType()))
        .withColumn("revisionnumber", F.col("revisionnumber").cast(T.IntegerType()))
        .withColumn("orderdate", F.to_timestamp("orderdate"))
        .withColumn("duedate", F.to_timestamp("duedate"))
        .withColumn("shipdate", F.to_timestamp("shipdate"))
        .withColumn("status", F.col("status").cast(T.IntegerType()))
        .withColumn("onlineorderflag", F.col("onlineorderflag").cast(T.BooleanType()))
        .withColumn("purchaseordernumber", F.col("purchaseordernumber").cast(T.StringType()))
        .withColumn("accountnumber", F.col("accountnumber").cast(T.StringType()))
        .withColumn("customerid", F.col("customerid").cast(T.LongType()))
        .withColumn("salespersonid", F.col("salespersonid").cast(T.LongType()))
        .withColumn("territoryid", F.col("territoryid").cast(T.LongType()))
        .withColumn("billtoaddressid", F.col("billtoaddressid").cast(T.LongType()))
        .withColumn("shiptoaddressid", F.col("shiptoaddressid").cast(T.LongType()))
        .withColumn("shipmethodid", F.col("shipmethodid").cast(T.LongType()))
        .withColumn("creditcardid", F.col("creditcardid").cast(T.LongType()))
        .withColumn("creditcardapprovalcode", F.col("creditcardapprovalcode").cast(T.StringType()))
        .withColumn("currencyrateid", F.col("currencyrateid").cast(T.LongType()))
        .withColumn("subtotal", F.col("subtotal").cast(T.DecimalType(18, 4)))
        .withColumn("taxamt", F.col("taxamt").cast(T.DecimalType(18, 4)))
        .withColumn("freight", F.col("freight").cast(T.DecimalType(18, 4)))
        .withColumn("totaldue", F.col("totaldue").cast(T.DecimalType(18, 4)))
        .withColumn("comment", F.col("comment").cast(T.StringType()))
        .withColumn("rowguid", F.col("rowguid").cast(T.StringType()))
        .withColumn("modifieddate", F.to_timestamp("modifieddate"))
    )


def rename_salesorderheader_columns(df: DataFrame) -> DataFrame:
    """
    Static column renaming for salesorderheader (camelCase -> snake_case).
    """
    rename_map = {
        "salesorderid": "sales_order_id",
        "revisionnumber": "revision_number",
        "orderdate": "order_date",
        "duedate": "due_date",
        "shipdate": "ship_date",
        "status": "status",
        "onlineorderflag": "online_order_flag",
        "purchaseordernumber": "purchase_order_number",
        "accountnumber": "account_number",
        "customerid": "customer_id",
        "salespersonid": "salesperson_id",
        "territoryid": "territory_id",
        "billtoaddressid": "bill_to_address_id",
        "shiptoaddressid": "ship_to_address_id",
        "shipmethodid": "ship_method_id",
        "creditcardid": "credit_card_id",
        "creditcardapprovalcode": "credit_card_approval_code",
        "currencyrateid": "currency_rate_id",
        "subtotal": "subtotal",
        "taxamt": "tax_amt",
        "freight": "freight",
        "totaldue": "total_due",
        "comment": "comment",
        "rowguid": "row_guid",
        "modifieddate": "modified_date",
    }

    out = df
    for old_name, new_name in rename_map.items():
        if old_name in out.columns:
            out = out.withColumnRenamed(old_name, new_name)
    return out


def sanity_clean_salesorderheader(df: DataFrame) -> DataFrame:
    """
    Light sanity cleaning for salesorderheader.
    """
    return (
        df
        # monetary fields should not be negative
        .withColumn("subtotal", F.when(F.col("subtotal") < 0, F.lit(0)).otherwise(F.col("subtotal")))
        .withColumn("taxamt", F.when(F.col("taxamt") < 0, F.lit(0)).otherwise(F.col("taxamt")))
        .withColumn("freight", F.when(F.col("freight") < 0, F.lit(0)).otherwise(F.col("freight")))
        .withColumn("totaldue", F.when(F.col("totaldue") < 0, F.lit(0)).otherwise(F.col("totaldue")))
        # order/ship/due dates should not be before orderdate (if they are, null them)
        .withColumn(
            "shipdate",
            F.when(
                F.col("shipdate").isNotNull() &
                F.col("orderdate").isNotNull() &
                (F.col("shipdate") < F.col("orderdate")),
                F.lit(None)
            ).otherwise(F.col("shipdate"))
        )
        .withColumn(
            "duedate",
            F.when(
                F.col("duedate").isNotNull() &
                F.col("orderdate").isNotNull() &
                (F.col("duedate") < F.col("orderdate")),
                F.lit(None)
            ).otherwise(F.col("duedate"))
        )
    )

In [0]:
sales_order_header_df = (
    spark
    .table(f"{CATALOG_BRONZE}.{SCHEMA_BRONZE}.sales_salesorderheader")
    .transform(cast_salesorderheader_types)
    .transform(sanity_clean_salesorderheader)
    .transform(lambda df: dedupe_by_column_as_id(df, "salesorderid"))
    .transform(rename_salesorderheader_columns)
    .write.mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.sales_order_header")
) 